<a href="https://colab.research.google.com/github/GreatLakesCommission/IEDRR_inland_lakes/blob/main/get_spp_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Script to request, QA/QC and combine invasive species observations from several sources.
Run the setup blocks, the blocks for the sources you want to pull from, then the combine block.

Requesting data from GBIF and EDDMapS requires that you have an account with them.


*   To use the GBIF code block, add your GBIF username and password and an email address for notifications to Secrets in your copy of this notebook as GBIF_USER, GBIF_PWD and GBIF_EMAIL.
*   To use the EDDMaps block, add your account credentials as EDDMAPS_USER and EDDMAPS_PWD.




**Setup Blocks**

In [1]:
# general setup
%%capture
from google.colab import drive
drive.mount('/content/drive')
import datetime
import os
os.chdir("drive/My Drive/iedrr")
today = datetime.date.today().strftime('%Y%m%d')
outfolder = "speciesobs_"+today
if not os.path.exists(outfolder):
  os.mkdir(outfolder)
os.chdir(outfolder)
import pandas as pd
!pip install pyreadr
import pyreadr
from scipy import spatial
from google.colab import userdata



In [2]:

# invasive species of interest, scientific names are formatted as lists to make it
# possible to include alternate names where more than one is in use across the
# regional databases

fishlist = [
    [['Channa'], 'Snakeheads'],
    [['Clarias batrachus'], 'Walking catfish'],
    [['Gymnocephalus cernua'], 'Ruffe'],
    [['Misgurnus anguillicaudatus'], 'Pond loach'],
    [['Neogobius melanostomus'], 'Round goby'],
    [['Osmerus mordax'], 'Rainbow smelt'],
    [['Osteoglossum bicirrhosum'], 'Silver arowana'],
    [['Proterorhinus semilunaris'], 'Tubenose goby'],
    [['Tinca tinca'], 'Tench'],
]

plantlist = [
    [['Alternanthera philoxeroides'], 'Alligator weed'],
    [['Cabomba caroliniana'], 'Carolina fanwort'],
    [['Callitriche stagnalis'], 'Pond water-starwort'],
    [['Elodea densa', 'Egeria densa'], 'Brazilian waterweed'],
    [['Hottonia palustris'], 'Water violet'],
    [['Hydrilla verticillata'], 'Hydrilla'],
    [['Hydrocharis morsus-ranae'], 'European frog-bit'],
    [['Hygrophila polysperma'], 'Indian swampweed'],
    [['Limnophila sessiliflora'], 'Dwarf ambulia'],
    [['Ludwigia grandiflora'], 'Large-flower primrose-willow'],
    [['Ludwigia hexapetala'], 'Six petal water primrose'],
    [['Ludwigia peploides'], 'Creeping water primrose'],
    [['Marsilea mutica'], 'Australian water-clover'],
    [['Marsilea quadrifolia'], 'European water-clover'],
    [['Myriophyllum aquaticum'], 'Parrot feather'],
    [['Najas minor'], 'Brittle naiad'],
    [['Nasturtium officinale'], 'Water-cress'],
    [['Nelumbo nucifera'], 'Sacred lotus'],
    [['Nitellopsis obtusa'], 'Starry stonewort'],
    [['Ottelia alismoides'], 'Duck-lettuce'],
    [['Pistia stratiotes'], 'Water lettuce'],
    [['Pontederia azurea', 'Eichhornia azurea'], 'Anchored water-hyacinth'],
    [['Pontederia crassipes', 'Eichhornia crassipes'],'Common water-hyacinth'],
    [['Sagittaria sagittifolia'], 'Hawaii arrowhead'],
    [['Salvinia auriculata'], 'Eared salvinia'],
    [['Salvinia molesta'], 'Giant salvinia'],
    [['Spirodela punctata'], 'Dotted duckweed'],
    [['Stratiotes aloides'], 'Water soldier'],
    [['Trapa natans'], 'European water chestnut']
]


# plantlist_slim doesn't include emergent species
plantlist_slim = [
    [['Cabomba caroliniana'], 'Carolina fanwort'],
    [['Callitriche stagnalis'], 'Pond water-starwort'],
    [['Elodea densa', 'Egeria densa'], 'Brazilian waterweed'],
    [['Hydrilla verticillata'], 'Hydrilla'],
    [['Hydrocharis morsus-ranae'], 'European frog-bit'],
    [['Hygrophila polysperma'], 'Indian swampweed'],
    [['Limnophila sessiliflora'], 'Dwarf ambulia'],
    [['Marsilea mutica'], 'Australian water-clover'],
    [['Marsilea quadrifolia'], 'European water-clover'],
    [['Myriophyllum aquaticum'], 'Parrot feather'],
    [['Najas minor'], 'Brittle naiad'],
    [['Nelumbo nucifera'], 'Sacred lotus'],
    [['Nitellopsis obtusa'], 'Starry stonewort'],
    [['Ottelia alismoides'], 'Duck-lettuce'],
    [['Pistia stratiotes'], 'Water lettuce'],
    [['Pontederia azurea', 'Eichhornia azurea'], 'Anchored water-hyacinth'],
    [['Pontederia crassipes', 'Eichhornia crassipes'],'Common water-hyacinth'],
    [['Sagittaria sagittifolia'], 'Hawaii arrowhead'],
    [['Salvinia auriculata'], 'Eared salvinia'],
    [['Salvinia molesta'], 'Giant salvinia'],
    [['Spirodela punctata'], 'Dotted duckweed'],
    [['Stratiotes aloides'], 'Water soldier'],
    [['Trapa natans'], 'European water chestnut']
]

invertlist = [
    [['Bithynia tentaculata'], 'Faucet snail'],
    [['Bythotrephes longimanus'], 'Spiny water flea'],
    [['Cercopagis pengoi'], 'Fishhook waterflea'],
    [['Corbicula fluminea'], 'Basket clam'],
    [['Dreissena bugensis'], 'Quagga mussel'],
    [['Dreissena polymorpha'], 'Zebra mussel'],
    [['Eriocheir sinensis'], 'Mitten crab'],
    [['Hemimysis anomala'], 'Bloody red shrimp'],
    [['Melanoides tuberculata'], 'Red-rimmed melania'],
    [['Potamopyrgus antipodarum'], 'New Zealand mud snail'],
    [['Procambarus virginalis'],'Marbled crayfish (Marmorkrebs)']
]



fishlist = pd.DataFrame(fishlist, columns=['sci_name', 'common_name'])
plantlist = pd.DataFrame(plantlist, columns=['sci_name', 'common_name'])
plantlist_slim = pd.DataFrame(plantlist_slim, columns=['sci_name', 'common_name'])
invertlist = pd.DataFrame(invertlist, columns=['sci_name', 'common_name'])

fishlist['taxon'] = 'fish'
plantlist['taxon'] = 'plant'
plantlist_slim['taxon'] = 'plant'
invertlist['taxon'] = 'invertebrate'



my_vars = {}

my_vars["fish"] = fishlist
my_vars["plant"] = plantlist
my_vars["invert"] = invertlist


# get institution lat/longs for QAQC
url = "https://github.com/ropensci/CoordinateCleaner/raw/refs/heads/master/data/institutions.rda"
dst_path = os.path.join(os.getcwd(), "institutions.rda") # download to CoLab
res = pyreadr.read_r(pyreadr.download_file(url, dst_path)) # convert rda to dictionary of dataframes

institutions = res["institutions"]
# filter to institutions within GL bounding box
institutions = institutions[(institutions['decimalLatitude'].between(36.9171,49.6117)) & (institutions['decimalLongitude'].between(-100.5513,-71.79))]
# set up a k-dimensional tree
inst_coords = list(zip(institutions["decimalLatitude"], institutions["decimalLongitude"]))
tree = spatial.KDTree(inst_coords)

def calculate_min(row):
    return tree.query([(row["lat_dec"],row["lon_dec"])])[0][0]

!rm institutions.rda

GBIF

In [3]:
#GBIF setup
!pip install pygbif
# capture suppresses output


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 464.5 kB/s eta 0:00:00


In [4]:

from pygbif import species as species
from pygbif import occurrences as occ
from pygbif.occurrences.download import GbifDownload
import os
import glob
import datetime
from time import sleep
import zipfile

# add your gbif.org username, password and contact email for download notices to Colab's 'Secrets'
# toggle notebook access on for all three
%env GBIF_USER=userdata.get('GBIF_USER')
%env GBIF_PWD = userdata.get('GBIF_PWD')
%env GBIF_EMAIL = userdata.get('GBIF_EMAIL')

SLEEP_DURATION = 20


# download GBIF species obs since 1970 within GL bounding box as zip files via API


skip = [] #skip taxa you previously exported a csv for

# get GBIF taxon IDs based on scientific names
def getskey(z):
  return species.name_backbone(z)['usageKey']

for i, (k, v) in enumerate(my_vars.items()):
  if not k in skip:
    records = []
    print("downloading ", k)
    splist = v['sci_name'].sum() # make list of all species inc. alternate scientific names
    #splist = list(set(splist))
    spkeys = [ getskey(x) for x in splist ]
    spkeys = list(map(str, spkeys))

    # construct query
    gbif_query = GbifDownload(userdata.get('GBIF_USER'), userdata.get('GBIF_EMAIL'))
    gbif_query.add_predicate_dict({"type": "in", "key": "BASIS_OF_RECORD", "values": ['HUMAN_OBSERVATION', 'OBSERVATION', 'MACHINE_OBSERVATION', 'LIVING_SPECIMEN', 'MATERIAL_SAMPLE'], "matchCase": "false"})
    gbif_query.add_predicate_dict({"type": "equals", "key": 'HAS_COORDINATE', 'value': 'TRUE', "matchCase": "false"})
    gbif_query.add_predicate_dict({"type": "equals", "key": 'HAS_GEOSPATIAL_ISSUE', 'value': 'FALSE', "matchCase": "false"})
    gbif_query.add_predicate_dict({"type": "within", "geometry": "POLYGON((-100.551 36.917,-71.79 36.917,-71.79 49.612,-100.551 49.612,-100.551 36.917))"})
    gbif_query.add_predicate_dict({"type": "greaterThanOrEquals", "key": 'YEAR', 'value': '1970', "matchCase": "false"})
    gbif_query.add_predicate_dict({"type": "in", "key": 'TAXON_KEY', 'values': spkeys, "matchCase": "false"})
    # submit download query
    xx = gbif_query.post_download(userdata.get('GBIF_USER'), userdata.get('GBIF_PWD'))
    # wait for download to be ready
    while True:
      print(f"waiting to get download {xx}...")
      status = occ.download_meta(key = xx)['status']

      if status not in ['PREPARING', 'RUNNING']:  # = not ready yet
          if status == 'SUCCEEDED':
              print(f"Download is ready, getting it")
              output_path = k+"_gbif_obs"
              if os.path.exists(output_path): # get rid of any previous downloads for this run
                files = glob.glob(output_path+'/*.zip')
                for f in files:
                  os.remove(f)
              else:
                os.mkdir(output_path)

              occ.download_get(xx, output_path)
          else:
              print(f"Status is {status}, why?")
              print(occ.download_meta(key = xx))
          break

      sleep(SLEEP_DURATION)

    print("finished with ", k)


env: GBIF_USER=userdata.get('GBIF_USER')
env: GBIF_PWD=userdata.get('GBIF_PWD')
env: GBIF_EMAIL=userdata.get('GBIF_EMAIL')
downloading  fish
waiting to get download 0000408-250117142028555...
Download is ready, getting it
finished with  fish
downloading  plant
waiting to get download 0000447-250117142028555...
waiting to get download 0000447-250117142028555...
waiting to get download 0000447-250117142028555...
waiting to get download 0000447-250117142028555...
waiting to get download 0000447-250117142028555...
waiting to get download 0000447-250117142028555...
waiting to get download 0000447-250117142028555...
Download is ready, getting it
finished with  plant
downloading  invert
waiting to get download 0000450-250117142028555...
waiting to get download 0000450-250117142028555...
waiting to get download 0000450-250117142028555...
waiting to get download 0000450-250117142028555...
waiting to get download 0000450-250117142028555...
Download is ready, getting it
finished with  invert


In [7]:

gbif_obs = {}
# read the GBIF data back in and QA/QC it
for k in my_vars:
  # read zip file into dataframe
  output_path = k+"_gbif_obs"
  files = os.listdir(output_path)
  file_path = os.path.join(output_path, files[0])
  print(file_path)
  base_name, extension = os.path.splitext(files[0])
  zf = zipfile.ZipFile(file_path)
  df = pd.read_csv(zf.open(base_name+'.csv'), sep='\t')

  print(len(df.index), k, " records")

  df = df[['gbifID', 'species', 'eventDate', 'decimalLatitude', 'decimalLongitude']]
  df.rename(columns={'gbifID':'uid', "species":"sci_name", "eventDate":"obs_date", "decimalLatitude":"lat_dec", "decimalLongitude":"lon_dec"}, inplace=True)
  df["source"] = "GBIF"
  df['obs_date'] = df['obs_date'].str.slice(0, 10)
  df['obs_date'] = pd.to_datetime(df['obs_date'], format='mixed')
  # Drop rows with invalid dates
  df = df.dropna(subset='obs_date')
  df.reset_index(drop=True,inplace=True)
  # export to csv before continuing because this cell will take forever
  df.to_csv(output_path + '/'+ k +'_obs_gbifraw_' + datetime.date.today().strftime('%Y%m%d') + '.csv', index=False)
  # add to dict
  gbif_obs[k] = df



fish_gbif_obs/0000408-250117142028555.zip


<ipython-input-7-5a190bec3e50>:11: DtypeWarning: Columns (17,29,36,37,38,39,40,41,43,44,46,48) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(zf.open(base_name+'.csv'), sep='\t')


49881 fish  records


<ipython-input-7-5a190bec3e50>:22: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('taxonKey').apply(lambda x: x.drop_duplicates(['decimalLatitude', 'decimalLongitude']))


plant_gbif_obs/0000447-250117142028555.zip


<ipython-input-7-5a190bec3e50>:11: DtypeWarning: Columns (39,46) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(zf.open(base_name+'.csv'), sep='\t')


19345 plant  records


<ipython-input-7-5a190bec3e50>:22: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('taxonKey').apply(lambda x: x.drop_duplicates(['decimalLatitude', 'decimalLongitude']))


invert_gbif_obs/0000450-250117142028555.zip
8874 invert  records


<ipython-input-7-5a190bec3e50>:22: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('taxonKey').apply(lambda x: x.drop_duplicates(['decimalLatitude', 'decimalLongitude']))


GLANSIS

In [9]:
import requests

# get GLANSIS's zipped csv of all data
url = "https://nas.er.usgs.gov/ipt/archive.do?r=nas_glansis"
filename = "GLANSIS_{}.zip".format(today)  # Choose a name for the downloaded file

response = requests.get(url)

if response.status_code == 200:
  with open(filename, "wb") as f:
    f.write(response.content)
  print("Zip file downloaded successfully.")
else:
  print("Failed to download the zip file.")
  # if dl fails, use most recent?
  #from pathlib import Path
  #DIR = Path("drive/My Drive/iedrr")
  #PATTERN = r'GLANSIS_*.zip'
  #latest_file = max(DIR.glob(PATTERN), key=lambda f: f.stat().st_ctime)

# open the zipped file
zf = zipfile.ZipFile(filename)
df = pd.read_csv(zf.open('occurrence.txt'), sep='\t')

df['eventDate'] = pd.to_datetime(df['eventDate'], format="%Y-%m-%d", errors='coerce')
# Drop rows with invalid dates
df = df.dropna(subset='eventDate')

#QAQC
df = df.loc[(df['decimalLatitude'] >= 36.917) & (df['decimalLatitude'] <= 49.612) & (df['decimalLongitude'] >= -100.551) & (df['decimalLongitude'] <= -71.79)] # bounding box
df = df.loc[df['eventDate']>datetime.datetime(1970,1,1)] # drop old data
df = df.loc[df['georeferenceRemarks'] != "Centroid"] # drop obs with poor coordinates
df.reset_index(drop=True,inplace=True)

glansis_obs = {}


for i, (k, v) in enumerate(my_vars.items()):
  splist = v['sci_name'].sum() # make list of all species inc. alternate scientific names
  hightax = []
  for item in splist:
    if len(item.split()) == 1:
      hightax.append(item)

  # drop nontarget species
  ndf = df[(df["scientificName"].isin(splist)) | (df["genus"].isin(hightax))]
  ndf.reset_index(drop=True)

  ndf = ndf[['id', 'scientificName', 'eventDate', 'decimalLatitude', 'decimalLongitude']]
  ndf.rename(columns={'id':'uid', "scientificName":"sci_name", "eventDate":"obs_date", "decimalLatitude":"lat_dec", "decimalLongitude":"lon_dec"}, inplace=True)
  ndf["source"] = "GLANSIS"
  glansis_obs[k] = ndf

Failed to download the zip file.


MISIN

In [13]:
!pip install esri2gpd
import esri2gpd


# MISIN observations layer updated daily
url = "https://services.arcgis.com/uHAHKfH1Z5ye1Oe0/arcgis/rest/services/misin_database_obs/FeatureServer/0"

misin_obs = {}

# convert esri date to datetime
def convert_esri_date(row):
    """Converts an esriFieldTypeDate value to a Python datetime object."""
    return datetime.datetime.fromtimestamp(row["DAY"] / 1000)  # Divide by 1000 to get seconds

for i, (k, v) in enumerate(my_vars.items()):
  # separate list of genera
  splist = v['sci_name'].sum() # make list of all species inc. alternate scientific names

  # separate out genera with no species epithet
  hightax = []
  for item in splist:
    if len(item.split()) == 1:
      hightax.append(item)
  if "Elodea densa" in splist:
    splist.append("Egeria densa") # MISIN is using Egeria densa instead of Elodea densa
  genus = [x.split()[0] for x in splist]

  gdf = esri2gpd.get(url, fields=['RECORDID', 'OBSERVER', 'DAY', 'LATITUDE', 'LONGITUDE', 'GENUS', 'SPECIES', 'VERIFIED'], where=f"GENUS IN {tuple(genus)}")
  gdf["DAY"] = gdf.apply(convert_esri_date, axis=1)
  #QAQC
  gdf = gdf[gdf["VERIFIED"] == 2]  # for field VERIFIED, 2 = 'Trusted Source', only keep these
  gdf = gdf.loc[(gdf['LATITUDE'] >= 36.917) & (gdf['LATITUDE'] <= 49.612) & (gdf['LONGITUDE'] >= -100.551) & (gdf['LONGITUDE'] <= -71.79)] # bounding box
  gdf = gdf.loc[gdf['DAY']>datetime.datetime(1970,1,1)] # drop old data
  # drop nontarget species
  gdf["sci_name"] = gdf["GENUS"] + " " + gdf["SPECIES"]
  gdf = gdf[(gdf["sci_name"].isin(splist)) | (gdf["GENUS"].isin(hightax))]
  gdf.reset_index(drop=True,inplace=True)
  gdf.drop(['geometry', 'OBSERVER', 'GENUS', 'SPECIES', 'VERIFIED'], axis=1, inplace=True)
  gdf.rename(columns={'RECORDID':'uid', "sci_name":"sci_name", "DAY":"obs_date", "LATITUDE":"lat_dec", "LONGITUDE":"lon_dec"}, inplace=True)
  gdf["source"] = "MISIN"
  gdf['uid'] = gdf['uid'].astype(str)

  misin_obs[k] = gdf

iMapInvasives

In [14]:
!pip install esri2gpd
import esri2gpd
# in GL states, iMapInvasives obs are almost exclusively in PA and NY

url = "https://imapinvasives.natureserve.org/arcgis/rest/services/public_presence/MapServer/4" # presences
url2 = "https://imapinvasives.natureserve.org/arcgis/rest/services/public_approximate_presence/MapServer/4" # approx coordinates presences

imap_obs = {}

import geopandas as gpd

# convert esri date to datetime
def convert_esri_date(row):
    """Converts an esriFieldTypeDate value to a Python datetime object."""
    return datetime.datetime.fromtimestamp(row["observation_date"] / 1000)  # Divide by 1000 to get seconds

for i, (k, v) in enumerate(my_vars.items()):
  # separate list of genera
  splist = v['sci_name'].sum() # make list of all species inc. alternate scientific names
  # separate out genera with no species epithet
  hightax = []
  for item in splist:
    if len(item.split()) == 1:
      hightax.append(item)

  genus = [x.split()[0] for x in splist]

  gdf = esri2gpd.get(url, fields=['present_species_id', 'observer_name', 'observation_date', 'jurisdiction', 'genus', 'scientific_name'], where=f"genus IN {tuple(genus)} AND jurisdiction IN ('New York', 'Pennsylvania', 'Michigan', 'Ohio')")
  gdf2 = esri2gpd.get(url2, fields=['present_species_id', 'observer_name', 'observation_date', 'jurisdiction', 'genus', 'scientific_name'], where=f"genus IN {tuple(genus)} AND jurisdiction IN ('New York', 'Pennsylvania', 'Michigan', 'Ohio')")
  rdf = gpd.GeoDataFrame(pd.concat([gdf, gdf2], ignore_index=True), crs=gdf.crs)

  rdf["observation_date"] = rdf.apply(convert_esri_date, axis=1)
  rdf["observation_date"] = pd.to_datetime(rdf["observation_date"], errors='coerce')
  #QAQC
  rdf = rdf.loc[rdf['observation_date']>datetime.datetime(1970,1,1)] # drop old data
  # drop nontarget species
  rdf = rdf[(rdf["scientific_name"].isin(splist)) | (rdf["genus"].isin(hightax))]
  rdf.reset_index(drop=True,inplace=True)
  rdf.rename(columns={'present_species_id':'uid', "scientific_name":"sci_name", "observation_date":"obs_date"}, inplace=True)
  rdf['uid'] = rdf['uid'].astype(str)
  rdf['lat_dec'] = rdf['geometry'].y.astype(float)
  rdf['lon_dec'] = rdf['geometry'].x.astype(float)
  rdf.drop(['geometry', 'observer_name', 'jurisdiction', 'genus'], axis=1, inplace=True)
  rdf["source"] = "iMapInvasives"
  # all obs in layer have been confirmed

  imap_obs[k] = rdf


/usr/local/lib/python3.11/dist-packages/esri2gpd/core.py:91: UserWarning: Long download time — total download will require 60 separate requests
  warnings.warn(


EDDMapS

In [15]:
from google.colab import userdata
import requests
import json
from pandas import json_normalize

edd_obs = {}

# log in to EDDMapS API
s = requests.session()

login_url = 'https://api.bugwoodcloud.org/v2/login'
headers = {
    "accept": "application/json",
    "Content-Type": "application/json"
}
resp = s.post(login_url, headers=headers, json={"email": userdata.get('EDDMAPS_USER'), "password": userdata.get('EDDMAPS_PWD')})

# resp.raise_for_status()

# can't search occurrence data using scientific names directly, have to get IDs first
subj_url = "https://api.bugwoodcloud.org/v2/subject"
sparams = dict()
sparams["searchon"] = "ScientificName"

for i, (k, v) in enumerate(my_vars.items()):
  print("starting ",k)

  namelist = []

  splist = v['sci_name'].sum() # make list of all species inc. alternate scientific names

  for name in splist:
    #print(name)
    # have to drop space between genus and species
    name = name.replace(" ", "")
    sparams["search"] = name
    r = s.get(subj_url, params=sparams)

    if r.status_code == 200:
      # Parse the JSON response
      data = json.loads(r.text)
      df = json_normalize(data)
      if df.empty:
        print("No results for",name)
        continue
      else:
        # specify fields to extract
        subfields = ["subjectid", "namepart1", "namepart2"]
        df = df[subfields]
        # clean up extra taxa that contain snakehead genus name
        if k == "fish":
          mask = df[df.apply(lambda row: row.str.contains('channa', case=False).any(), axis=1)]
          if not mask.empty:
            mask = mask.loc[mask['namepart1'] != "Channa"]
            df = pd.merge(df,mask, indicator=True, how='outer') \
              .query('_merge=="left_only"') \
              .drop('_merge', axis=1)
        #print (df)
        namelist.append(df)

    else:
        print("Error: ", r.status_code)

  namelist = pd.concat(namelist, ignore_index=True)

  subjs = ','.join(namelist['subjectid'].astype(str)) #comma-separated list of subject ids for species of interest

  # get occurrence data
  occ_url = 'https://api.bugwoodcloud.org/v2/occurrence'
  params = dict()
  params["subjectid"] = subjs
  # "scientificname" is not a valid param, need to use subjectids
  params["sortorder"] = "desc"
  params["enddate"] = "01/17/2025"
  params["startdate"] = "01/01/1970"
  params["paging"] = "false"

  statedfs = []

  for state in ["17","18","26","27","36","39","42","55"]: # FIPS codes for IL, IN, MI, MN, NY, OH, PA, WI
    params["state"] = state
    r2 = s.get(occ_url, params=params) # is there a limit to number of records?

    # Check if the request was successful
    if r2.status_code == 200:
      # Parse the JSON response
      data = json.loads(r2.text)
      df = json_normalize(data)
      # specify fields to extract
      subfields = ["objectid", "scientificname", "observationdate", "latitude_decimal", "longitude_decimal", "recordbasis", "identificationcredibility"]
      if not df.empty:
        df = df[subfields]
        statedfs.append(df)
      else:
        print("No ",k," results for ",state)

    else:
        print("Error: ", r.status_code)

  cdf = pd.concat(statedfs, ignore_index=True)
  # QAQC
  cdf = cdf[cdf.recordbasis != "Preserved Specimen"]
  keeps = ["Credible", "Verified"]
  cdf = cdf[cdf['identificationcredibility'].isin(keeps)]
  cdf.dropna(inplace=True)
  cdf.reset_index(drop=True,inplace=True)
  cdf.drop(['recordbasis', 'identificationcredibility'], axis=1, inplace=True)
  cdf.rename(columns={'objectid':'uid', "scientificname":"sci_name", "observationdate":"obs_date", "latitude_decimal":"lat_dec", "longitude_decimal":"lon_dec"}, inplace=True)
  cdf["source"] = "EDDMapS"
  cdf['uid'] = cdf['uid'].astype(str)
  cdf['obs_date'] = pd.to_datetime(cdf['obs_date'])
  print (cdf.shape[0],k," records")

  edd_obs[k] = cdf



starting  fish
No  fish  results for  39
No  fish  results for  42
25 fish  records
starting  plant
No results for Pontederiaazurea
44 plant  records
starting  invert
50 invert  records


Combine

In [103]:
count_lists = []
for k in my_vars:
  print(k)
  # combine obs from all sources
  dflist = [gbif_obs[k], glansis_obs[k], imap_obs[k], edd_obs[k], misin_obs[k]]
  for x in dflist:
    if pd.api.types.is_dtype_equal(x['obs_date'].dtype, "datetime64[ns, UTC]"):
      x['obs_date'] = x['obs_date'].dt.tz_localize(None)
  x['obs_date'] = x['obs_date'].dt.normalize()
    #print(x['obs_date'].dtype)
  df = pd.concat(dflist, ignore_index=True)
  df.reset_index(drop=True, inplace=True)

  # use consistent scientific name for Brazilian waterweed, water hyacinths todo: generalize this
  df.replace({'sci_name':{'Elodea densa':'Egeria densa', 'Eichhornia crassipes':'Pontederia crassipes', 'Eichhornia azurea':'Pontederia azurea'}}, inplace = True)

  # final QAQC before counts
  #drop observations whose coordinates are the location of an institution
  rcnt = df.shape[0]
  print(rcnt, " recs before QAQC")
  df["mindist"] = df.apply(calculate_min, axis=1)
  df = df[df["mindist"] > 0.00001] # buffer in degrees
  df.reset_index(drop=True, inplace=True)
  rcnt2 = df.shape[0]
  if rcnt2 < rcnt:
    print(rcnt - rcnt2, " recs at institution lat/longs were dropped")
  # drop obs if lat and long are the same
  df = df[df['lat_dec'] != df['lon_dec']]
  rcnt3 = df.shape[0]
  if rcnt3 < rcnt2:
    print(rcnt2 - rcnt3, " recs where lat = lon were dropped")
  df.dropna(inplace=True)
  rcnt4 = df.shape[0]
  if rcnt4 < rcnt3:
    print(rcnt3 - rcnt4, " recs with nans were dropped")
  print(rcnt4, " recs after QAQC")
  # total recs from each source
  for category, count in df['source'].value_counts().items():
      print(f"{count} total recs from {category}")
      row = [k,category, 'total', count]
      count_lists.append(row)
  # drop duplicates, with a margin of error to account for coordinate rounding
  from sklearn.cluster import AgglomerativeClustering

  distance = 0.0001
  xy = df[['lon_dec','lat_dec']].values.tolist()
  cluster = AgglomerativeClustering(n_clusters=None, linkage='single', metric='euclidean', distance_threshold=distance)
  cluster.fit(xy)
  df['group'] = cluster.labels_

  # combine group id, species, date into one microgroup column
  cols = ['group', 'sci_name', 'obs_date']
  df['microgroup'] = df[cols].apply(lambda row: '_'.join(row.values.astype(str)), axis=1)
  # Create a new column with the count of each microgroup
  # if count(microgroup)>1, mark as duplicate
  df['mg_count'] = df.groupby('microgroup')['microgroup'].transform('count')
  df['is_duplicate'] = df['mg_count'] > 1
  # how many recs does each source have that aren't in the other sources
  dfu = df[df['is_duplicate'] != True]
  for category, count in dfu['source'].value_counts().items():
      print(f"{count} unique recs from {category}")
      row = [k,category, 'unique', count]
      count_lists.append(row)
  print(df['is_duplicate'].value_counts())
  print('unique recs: ',df['microgroup'].nunique())
  # Within each microgroup keep the first version of that observation
  df = df.groupby('microgroup', as_index=False).first()
  df.reset_index(drop=True,inplace=True)
  df.drop(['group', 'microgroup', 'mg_count', 'mindist', 'is_duplicate'], axis=1, inplace=True)

  df["taxon"] = k
  # export final obs table for taxon to Drive folder
  cwd = os.getcwd()
  outfile = os.path.join(cwd, k + '_obs_allsources_' + datetime.date.today().strftime('%Y%m%d') + '.csv')
  df.to_csv(outfile, index=False)
  print("exported to ",outfile)


# save total and unique observation counts for each source
count_headers = ['taxon','source', 'type', 'count']
counts_df = pd.DataFrame(count_lists,columns=count_headers)
#counts_df = counts_df.to_frame().reset_index()
allcounts = os.path.join(cwd, 'obs_counts_by_taxon_and_source_' + datetime.date.today().strftime('%Y%m%d') + '.csv')
counts_df.to_csv(allcounts, index=False)

counts_df.drop(['taxon'], axis=1, inplace=True)

counts_df = counts_df.groupby(['source', 'type'])['count'].sum()
counts_df = counts_df.to_frame().reset_index()

counts_df = counts_df.sort_values(by=['source', 'type'])
countfile = os.path.join(cwd, 'obs_counts_by_source_' + datetime.date.today().strftime('%Y%m%d') + '.csv')
counts_df.to_csv(countfile, index=False)
print("done!")

fish
17386  recs before QAQC
17386  recs after QAQC
10169 total recs from GBIF
4127 total recs from GLANSIS
2002 total recs from iMapInvasives
1063 total recs from MISIN
25 total recs from EDDMapS
9940 unique recs from GBIF
3222 unique recs from GLANSIS
1455 unique recs from iMapInvasives
285 unique recs from MISIN
1 unique recs from EDDMapS
is_duplicate
False    14903
True      2483
Name: count, dtype: int64
unique recs:  15884
exported to  /content/drive/MyDrive/iedrr/speciesobs_20250118/fish_obs_allsources_20250118.csv
plant
67548  recs before QAQC
67548  recs after QAQC
31686 total recs from iMapInvasives
18848 total recs from GBIF
11318 total recs from MISIN
5652 total recs from GLANSIS
44 total recs from EDDMapS
24179 unique recs from iMapInvasives
11320 unique recs from GBIF
4129 unique recs from MISIN
466 unique recs from GLANSIS
43 unique recs from EDDMapS
is_duplicate
False    40137
True     27411
Name: count, dtype: int64
unique recs:  50540
exported to  /content/drive/MyDri